# Headline numbers + corpus audit

Stage 5 of the 6-stage producer chain — the final stage. Loads the YAML and parquet written by stage 04 and prints two pieces:

1. **Canonical `HEADLINE` sheet** — `prompt_analysis.headline_numbers(data, alt_df=…, parquet=…)`. Same call signature consumer notebooks use.
2. **Audit table** — the human-readable summary of corpus statistics (Files / Sentences / Word tokens / ccVersions / Rule sentences / etc.). Each row references the canonical YAML key its value comes from.

This is the only producer stage that appears in the published Quarto site (it's the most reader-friendly view); stages 00–04 run in the kernel but are hidden from the navbar.

## 12. Canonical HEADLINE sheet

Re-uses `prompt_analysis.headline_numbers()` — the same function consumer notebooks call. Pass `alt_df` (for composite-directiveness range and per-version `mood_marker_pct` extremes) and the per-sentence parquet (for parquet-level threat / causal / rule counts) so the producer's audit covers the full HEADLINE contract.

In [1]:
"""Compute and display the canonical HEADLINE dict.

Prints as YAML so the values are visible in this notebook (saving a copy in the
cell output) without requiring another tool.
"""
import os, sys, pathlib, importlib
sys.path.insert(0, ".")
import pandas as pd
import yaml as _yaml

_here = pathlib.Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [_here, *_here.parents] if (p / "prompt_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        f"Could not find prompt_pipeline.py walking up from {_here}. "
        "Run from inside the claude-prompts-analysis repo."
    )
if pathlib.Path.cwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

import prompt_analysis
importlib.reload(prompt_analysis)  # pick up edits without restarting the kernel
from prompt_analysis import load_yaml, build_alt_df, headline_numbers

data    = load_yaml()
alt_df  = build_alt_df(data)
parquet = pd.read_parquet("sentences_classified.parquet")

HEADLINE = headline_numbers(data, alt_df=alt_df, parquet=parquet)

print(_yaml.safe_dump(HEADLINE, sort_keys=False, default_flow_style=False))

n_files: 289
n_sentences: 5878
n_word_tokens: 133526
n_versions: 57
n_rule_sentences: 2285
pct_explained_same: 6.6521
pct_explained_para: 24.2888
judgment_count: 78
procedural_count: 594
judgment_to_procedural_ratio: 0.131
threat_count: 8
causal_count: 136
threat_share: 0.0556
question_count: 87
apology_count: 3
selfref_claude: 521
selfref_assistant: 20
selfref_model: 266
pct_anthropomorphic: 0.6456
positive_evaluative_quality: 296
positive_evaluative_emphasis: 184
positive_evaluative_union: 480
negative_evaluative: 152
ratio_quality_to_negative: 1.9473684210526316
ratio_union_to_negative: 3.1578947368421053
appreciative_sent: 4
collaborative_sent: 30
streak_max: 12
n_streaks_ge3: 230
n_streaks_ge5: 52
vocab_hard_prohibitions: 630
vocab_hard_prescriptions: 358
vocab_pronouns_2p: 1393
vocab_pronouns_1p: 185
vocab_profanity: 0
modality_deontic: 261
modality_epistemic: 321
modality_dynamic: 557
mood_marker_pct: 0.7676
top_caps_imperative:
- - IMPORTANT
  - 35
- - NEVER
  - 26
- - MUST
  -

## Audit table

Live corpus statistics — these are the canonical values for every prose mention across the notebooks; any number that disagrees gets reconciled to these.

| Quantity | Value |
|---|---:|
| Files | **289** |
| Sentences | **5,878** |
| Word tokens | **133,526** |
| ccVersions (distinct) | **57** (oldest `2.0.14`, latest `2.1.132`, mode `2.1.53` with 47 files) |
| Rule sentences | **2,285** |
| `pct_explained_same` | **6.65%** |
| `pct_explained_para` | **24.29%** (round to 24.3% in narrative prose) |
| `judgment_count` / `procedural_count` / ratio | **78 / 594 / 0.131** |
| `threat_count` / `causal_count` / `threat_share` | **8 / 136 / 0.0556** (5.6% in narrative) |
| `soft_conditional_count` (procedural connectives, reported separately) | **96** — see [`docs/THREAT_CLASSIFIER.md`](docs/THREAT_CLASSIFIER.md) for the two-tier split |
| `question_count` / `apology_count` | 87 / 3 |
| `selfref_claude` / `_assistant` / `_model` | 521 / 20 / 266 |
| `pct_anthropomorphic` | **0.6456** (64.6%) |
| Imperative streaks: `streak_max` / `n_ge3` / `n_ge5` | 12 / 230 / 52 |
| RULES-section paragraphs (in / out, % explained) | 27 (18.52%) / 1,284 (16.51%) |
| Modality (deontic / epistemic / dynamic) | 261 / 321 / 557 |
| `vocab.hard_prohibitions.count` | **630** |
| `vocab.hard_prescriptions.count` | 358 |
| `vocab.pronouns_2p.count` | **1,393** |
| `vocab.pronouns_1p.count` | **185** |
| `vocab.profanity.count` | 0 |
| Stance: positive_evaluative_quality / _emphasis / negative_evaluative | **296 / 184 / 152** |
| Quality-only positive-vs-negative ratio | **1.95×** (296 / 152) |
| Union positive-vs-negative ratio | **3.16×** (480 / 152) |
| `appreciative_sent_count` | **4** |
| `collaborative_sent_count` | **30** |